In [1]:
from march.module import DMC
from march.ops.grid import create_voxel_grid, drop_grid_w_mesh

import torch
import open3d as o3d
import open3d.core as o3c
import numpy as np
import plotly.graph_objects as go

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Object

In [2]:
sphere_mesh = o3d.t.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)

print("# vertices:", len(sphere_mesh.vertex.positions))
print("# triangles:", len(sphere_mesh.triangle.indices))

# vertices: 762
# triangles: 1520


# Voxel Grid

In [3]:
resolution = 8
bounds = [[-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1]]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
grid_vertices, cubes = create_voxel_grid(
    res_x=resolution,
    res_y=resolution,
    res_z=resolution,
    bounds=bounds,
)

print("grid vertices:", grid_vertices.shape)
print("cubes:", cubes.shape)

grid vertices: (729, 3)
cubes: (512, 8)


# Sign Distance

In [5]:
scene = o3d.t.geometry.RaycastingScene()
scene.add_triangles(sphere_mesh)

iso = 0.0

In [6]:
queries = o3c.Tensor(grid_vertices, dtype=o3c.float32)
print("queries:", queries.shape)

queries: SizeVector[729, 3]


In [7]:
distances = scene.compute_signed_distance(queries)

print("distances:", distances.shape)
print("distances (min, max):", distances.min().item(), distances.max().item())

distances: SizeVector[729]
distances (min, max): -0.9938629269599915 0.9054293632507324


In [8]:
print(type(distances))

<class 'open3d.cuda.pybind.core.Tensor'>


In [9]:
fig = go.Figure()

# Create color array: red if value > iso, blue otherwise
colors = ['red' if v > iso else 'blue' for v in distances.cpu().numpy()]

# Add grid points as scatter plot
fig.add_trace(go.Scatter3d(
    x=grid_vertices[:, 0],
    y=grid_vertices[:, 1],
    z=grid_vertices[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    name='Grid Points'
))

# Reconstruction

In [10]:
grid_vertices = torch.from_numpy(grid_vertices).to(device)
cubes = torch.from_numpy(cubes).to(device)
distances = torch.from_numpy(distances.cpu().numpy()).to(device)

In [11]:
print(f"grid_vertices dtype: {grid_vertices.dtype}, device: {grid_vertices.device}")
print(f"cubes dtype: {cubes.dtype}, device: {cubes.device}")
print(f"distances dtype: {distances.dtype}, device: {distances.device}")

grid_vertices dtype: torch.float32, device: cuda:0
cubes dtype: torch.int32, device: cuda:0
distances dtype: torch.float32, device: cuda:0


In [12]:
mesh_vertices = torch.from_numpy(sphere_mesh.vertex.positions.cpu().numpy()).to(device)
mesh_faces = torch.from_numpy(sphere_mesh.triangle.indices.cpu().numpy()).to(device)

print(f"mesh_vertices dtype: {mesh_vertices.dtype}, device: {mesh_vertices.device}")
print(f"mesh_faces dtype: {mesh_faces.dtype}, device: {mesh_faces.device}")

mesh_vertices dtype: torch.float32, device: cuda:0
mesh_faces dtype: torch.int64, device: cuda:0


In [13]:
drop_grid_vertices, drop_grid_cubes, unique_vertex_indices = drop_grid_w_mesh(
    grid_vertices=grid_vertices,
    grid_cubes=cubes,
    mesh_vertices=mesh_vertices,
    mesh_faces=mesh_faces,
)

print("drop_grid_vertices:", drop_grid_vertices.shape)
print("drop_grid_cubes:", drop_grid_cubes.shape)
print("unique_vertex_indices:", unique_vertex_indices.shape)

# before 
# before 512 cubes -> after 304 cubes
# before 729 vertices -> after 564 vertices

torch.int32
drop_grid_vertices: torch.Size([564, 3])
drop_grid_cubes: torch.Size([304, 8])
unique_vertex_indices: torch.Size([564])


In [14]:
drop_distances = distances[unique_vertex_indices]

print("drop_distances:", drop_distances.shape)

drop_distances: torch.Size([564])


In [15]:
fig = go.Figure()

# Create color array: red if value > iso, blue otherwise
colors = ['red' if v > iso else 'blue' for v in drop_distances.cpu().numpy()]

drop_grid_vertices_np = drop_grid_vertices.cpu().numpy()
print("drop_grid_vertices_np:", drop_grid_vertices_np.shape)

# Add grid points as scatter plot
fig.add_trace(go.Scatter3d(
    x=drop_grid_vertices_np[:, 0],
    y=drop_grid_vertices_np[:, 1],
    z=drop_grid_vertices_np[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    name='Grid Points'
))

drop_grid_vertices_np: (564, 3)


In [16]:
print(f"drop_grid_vertices dtype: {drop_grid_vertices.dtype}, device: {drop_grid_vertices.device}")
print(f"cubes dtype: {drop_grid_cubes.dtype}, device: {drop_grid_cubes.device}")
print(f"distances dtype: {drop_distances.dtype}, device: {drop_distances.device}")

drop_grid_vertices dtype: torch.float32, device: cuda:0
cubes dtype: torch.int32, device: cuda:0
distances dtype: torch.float32, device: cuda:0


In [17]:
mc = DMC()

vertices, triangles = mc(
    grid_vertices=drop_grid_vertices, 
    cubes=drop_grid_cubes, 
    values=drop_distances, 
    iso=iso
)

In [18]:
print(f"# vertices: {vertices.shape[0]}, # triangles: {triangles.shape[0]}")
print(f"vertices dtype: {vertices.dtype}, device: {vertices.device}")
print(f"triangles dtype: {triangles.dtype}, device: {triangles.device}")

# vertices: 270, # triangles: 536
vertices dtype: torch.float32, device: cuda:0
triangles dtype: torch.int64, device: cuda:0


In [ ]:
# Create figure
fig = go.Figure()

grids_numpy = grid_vertices.detach().cpu().numpy()
verts_numpy = vertices.detach().cpu().numpy()
faces_numpy = triangles.detach().cpu().numpy()

# Add mesh as wireframe
# First, create the mesh surface with low opacity
fig.add_trace(go.Mesh3d(
    x=verts_numpy[:, 0],
    y=verts_numpy[:, 1],
    z=verts_numpy[:, 2],
    i=faces_numpy[:, 0],
    j=faces_numpy[:, 1],
    k=faces_numpy[:, 2],
    opacity=0.1,
    color='lightblue',
    showlegend=False
))

# Add wireframe edges
edges_x = []
edges_y = []
edges_z = []
for i, j, k in faces_numpy:
    # Edge 1: vertex i to j
    edges_x.extend([verts_numpy[i, 0], verts_numpy[j, 0], None])
    edges_y.extend([verts_numpy[i, 1], verts_numpy[j, 1], None])
    edges_z.extend([verts_numpy[i, 2], verts_numpy[j, 2], None])
    # Edge 2: vertex j to k
    edges_x.extend([verts_numpy[j, 0], verts_numpy[k, 0], None])
    edges_y.extend([verts_numpy[j, 1], verts_numpy[k, 1], None])
    edges_z.extend([verts_numpy[j, 2], verts_numpy[k, 2], None])
    # Edge 3: vertex k to i
    edges_x.extend([verts_numpy[k, 0], verts_numpy[i, 0], None])
    edges_y.extend([verts_numpy[k, 1], verts_numpy[i, 1], None])
    edges_z.extend([verts_numpy[k, 2], verts_numpy[i, 2], None])

fig.add_trace(go.Scatter3d(
    x=edges_x,
    y=edges_y,
    z=edges_z,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name='Mesh Edges',
    showlegend=True
))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='Grid Points and Marching Cubes Mesh (Wireframe)',
    width=800,
    height=800
)

fig.show()


In [20]:
# Export to OBJ

o3d_mesh = o3d.geometry.TriangleMesh()
o3d_mesh.vertices = o3d.utility.Vector3dVector(verts_numpy)
o3d_mesh.triangles = o3d.utility.Vector3iVector(faces_numpy)
o3d.io.write_triangle_mesh("marching_cubes_sphere.obj", o3d_mesh)

True